In [37]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
print(f"GPU available: {torch.cuda.is_available()}")


GPU available: True


In [38]:
df = pd.read_csv("dataset/vibe_coding_search.csv")

df = df.drop_duplicates(subset=['ID'])
df = df.drop_duplicates(subset=['Text'])

df = df.dropna(subset=['Text'])

df['Clean_Text'] = df['Text'].str.lower()

print(f"Total unique records before filtering: {len(df)}")

Total unique records before filtering: 30550


In [39]:
job_keywords = [
    'hiring', 'apply now', 'salary', 'years of experience', 
    'job requirement', 'qualifications', 'full-time', 'benefits', 
    'recruiting', 'competitive pay', 'join our team', 'AI Accelerator'
]

pattern = '|'.join(job_keywords)

df = df[~df['Clean_Text'].str.contains(pattern, case=False, na=False)]

print(f"Records remaining after Keyword Blacklist: {len(df)}")

Records remaining after Keyword Blacklist: 29414


In [40]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ideal sentence 
target_query = "vibe coding, using AI agents like Cursor or Github Copilot to write software, AI assisted programming, LLM code generation"

# Encode the target query into a mathematical vector space
target_embedding = model.encode(target_query, convert_to_tensor=True)

print("Model and target embedding loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1126.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and target embedding loaded!


In [41]:
# Batched processing to be faster 
corpus_embeddings = model.encode(df['Clean_Text'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True)

# cos similarity betw ideal sentence and text
cosine_scores = util.cos_sim(target_embedding, corpus_embeddings)[0]
df['Relevance_Score'] = cosine_scores.cpu().numpy()

print("\nTop 3 Most Relevant Posts:")
print(df.nlargest(3, 'Relevance_Score')[['Text', 'Relevance_Score']])

Batches: 100%|██████████| 460/460 [00:12<00:00, 37.81it/s] 


Top 3 Most Relevant Posts:
                                                   Text  Relevance_Score
2526  What’s the best tool for non-techies starting ...         0.758611
2616  50 AI Vibe Coding Tools for Everyone in 2025 *...         0.732192
6899  The Ultimate List of AI Tools for Vibe Coding ...         0.717682


In [42]:
# Function to calculate total words in a dataframe
def get_total_words(dataframe):
    return dataframe['Clean_Text'].apply(lambda x: len(str(x).split())).sum()

threshold = 0.40
step = 0.02

while threshold > 0.10:
    # Filter the dataframe based on the current threshold
    df_filtered = df[df['Relevance_Score'] >= threshold]
    
    total_records = len(df_filtered)
    total_words = get_total_words(df_filtered)
    
    # Check assignment constraints
    if total_records >= 10000 and total_words >= 100000:
        print(f"Extracted threshold: {threshold:.2f}")
        print(f"Final Records: {total_records} (Target: 10,000+)")
        print(f"Final Words: {total_words} (Target: 100,000+)")
        break
    else:
        # if fails, threshold must be lowered
        threshold -= step

Extracted threshold: 0.18
Final Records: 10665 (Target: 10,000+)
Final Words: 1973416 (Target: 100,000+)


In [43]:
df_final = df_filtered.drop(columns=['Relevance_Score'])

df_final.to_csv("dataset/vibe_coding_transformer_processed.csv", index=False)